# Build the Merge Tree Yourself

**Topic:** Agglomerative clustering &nbsp;·&nbsp; **Difficulty:** Hard

A dendrogram is not magic: it is `n-1` merges, each recording which two clusters
joined and at what distance. Once you have that list, "cut the tree at k
clusters" is just "undo the last `k-1` merges" (re-running the merges and stopping k-1 steps early is enough, and grading needs only the heights).

Implement average linkage from scratch (no `scipy`, no `sklearn`) and read the
cluster assignment straight off your own merge list. The largest gap between
consecutive merge heights is the usual heuristic for choosing k, and here it
gets the right answer.

## The data (created in the next cell)

- `X` `(60, 3)`: four compact, well-separated groups of 15 points.
- Small on purpose: an `O(n^3)` implementation is fine at this size.

## Your task

With **NumPy only**:

1. Start with 60 singleton clusters. Repeatedly merge the two clusters with the
   smallest **average** inter-point distance (the mean over all cross pairs),
   recording the merge height each time, until one cluster remains.
2. `heights`: the 59 merge heights in the order they occurred.
3. `labels_k4`: the cluster assignment obtained by cutting so that 4 clusters
   remain, relabelled so that clusters appear in order of their smallest member
   index (cluster of point 0 is 0, and so on).
4. `best_k`, the k suggested by the largest jump in the last 10 merge heights:
   with `tail = heights[-10:]` and `gaps = np.diff(tail)`,
   `best_k = len(tail) - argmax(gaps)`, the number of clusters left once every
   merge below the largest gap has been undone.
5. Set `answers` to a dict with `"heights"` `(59,)`, `"labels_k4"` `(60,)`,
   `"n_clusters_k4"` (int), `"best_k"` (int).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks your 4-cluster cut recovers the four planted groups exactly.

In [ ]:
import numpy as np

rng = np.random.default_rng(1919)
centres = np.array([[0, 0, 0], [8, 0, 0], [0, 8, 0], [0, 0, 8]], dtype=float)
X = np.vstack([c + rng.normal(0, 0.45, (15, 3)) for c in centres])
y_true = np.repeat(np.arange(4), 15)
print(f"X {X.shape}, 4 planted groups of 15")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("heights", [59], 64.52506681665534, -8.20928630334287, 1e-06),
    ("labels_k4", [60], 90.0, -20.071271572539747, 1e-06),
    ("n_clusters_k4", [], 4.0, -0.6697520592579966, 1e-06),
    ("best_k", [], 4.0, -0.6697520592579966, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "labels_k4" in answers:
    from sklearn.metrics import adjusted_rand_score
    _ari = adjusted_rand_score(y_true, np.asarray(answers["labels_k4"], dtype=int))
    print(f"  {'cut recovers 4 groups':<24} {'ok' if _ari > 0.999 else 'wrong'}  (ARI = {_ari:.4f})")
    _ok = _ok and _ari > 0.999

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Keeping the members of each live cluster in a list and recomputing the average
cross-distance is `O(n^3)` overall, which is nothing at n = 60 and is the
clearest way to see what "average linkage" means. Cutting at k is bookkeeping:
run the merges but stop `k-1` steps early. The largest gap in merge heights marks
the point where you started joining things that are genuinely far apart, the
same signal a dendrogram shows as a tall vertical stretch with no merges.

```python
import numpy as np

n = len(X)
D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))

def run(stop_at):
    members = {i: [i] for i in range(n)}
    heights = []
    while len(members) > stop_at:
        keys = sorted(members)
        best, bi, bj = np.inf, None, None
        for ai in range(len(keys)):
            for bj_ in range(ai + 1, len(keys)):
                p, q = keys[ai], keys[bj_]
                d = float(D[np.ix_(members[p], members[q])].mean())
                if d < best:
                    best, bi, bj = d, p, q
        heights.append(best)
        members[bi] = members[bi] + members[bj]
        del members[bj]
    return members, heights

_, heights_all = run(1)
heights = np.array(heights_all)

members4, _ = run(4)
labels = np.empty(n, dtype=int)
for new_id, key in enumerate(sorted(members4, key=lambda k: min(members4[k]))):
    labels[members4[key]] = new_id

tail = heights[-10:]
gaps = np.diff(tail)
best_k = int(len(tail) - int(np.argmax(gaps)))

answers = {
    "heights": heights,
    "labels_k4": labels,
    "n_clusters_k4": int(len(np.unique(labels))),
    "best_k": best_k,
}
```